In [ ]:
import os
import cv2
import fitz  # PyMuPDF
import numpy as np
from doclayout_yolo import YOLOv10

# Paths
pdf_path = "AMER_A&D_Defense_L3Harris_Q1 (Jan-Mar).pdf"
output_dir = "annotated_pages_AMER"

os.makedirs(output_dir, exist_ok=True)

# Load model
model = YOLOv10("doclayout_yolo_docstructbench_imgsz1024.pt")

# Open PDF
doc = fitz.open(pdf_path)

print(f"Total pages: {len(doc)}")

for i, page in enumerate(doc):
    print(f"Processing page {i+1}...")

    # Render page to image (increase matrix for better resolution)
    zoom = 2  # 2 = ~300 DPI
    mat = fitz.Matrix(zoom, zoom)
    pix = page.get_pixmap(matrix=mat)

    # Convert to numpy array
    img = np.frombuffer(pix.samples, dtype=np.uint8).reshape(pix.height, pix.width, pix.n)

    # Handle RGB / RGBA
    if pix.n == 4:
        img = cv2.cvtColor(img, cv2.COLOR_RGBA2BGR)
    else:
        img = cv2.cvtColor(img, cv2.COLOR_RGB2BGR)

    # Run YOLO
    det_res = model.predict(
        img,
        imgsz=1024,
        conf=0.2,
        device="cpu"
    )

    # Annotate
    annotated = det_res[0].plot(pil=False, line_width=2, font_size=8)

    # Save
    output_path = os.path.join(output_dir, f"page_{i+1:03d}.jpg")
    cv2.imwrite(output_path, annotated)

print("Done! All annotated pages saved.")

In [1]:
# =========================
# PDF -> YOLO layout -> OpenAI -> Markdown (parallel)
# Reading order handled by prompt only
# =========================

import os
import cv2
import fitz  # PyMuPDF
import base64
import threading
import numpy as np
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor, as_completed
from portkey_ai import Portkey
from doclayout_yolo import YOLOv10

# -------------------------
# Config
# -------------------------
PDF_PATH = r"..\Sample PDFs\InfoEdge_Annual_Report_2025.pdf"
YOLO_WEIGHTS = "doclayout_yolo_docstructbench_imgsz1024.pt"
OUTPUT_DIR = "markdown_output_InfoEdge_Annual_Report_2025"
MODEL_NAME = "gpt-5.4-mini"
MAX_WORKERS = 16
YOLO_IMGSZ = 1024
YOLO_CONF = 0.20
ZOOM = 2.0
MAX_BLOCKS_PER_PAGE = 40

os.makedirs(OUTPUT_DIR, exist_ok=True)

# -------------------------
# Client
# -------------------------
client = Portkey(
    base_url=os.environ.get("PORTKEY_BASE_URL"),
    api_key=os.environ.get("PORTKEY_API_KEY")
)

# -------------------------
# Thread-local YOLO model
# -------------------------
_thread_local = threading.local()

def get_layout_model():
    if not hasattr(_thread_local, "model"):
        _thread_local.model = YOLOv10(YOLO_WEIGHTS)
    return _thread_local.model

# -------------------------
# Helpers
# -------------------------
def page_to_bgr(page, zoom=2.0):
    mat = fitz.Matrix(zoom, zoom)
    pix = page.get_pixmap(matrix=mat, alpha=False)
    img = np.frombuffer(pix.samples, dtype=np.uint8).reshape(pix.height, pix.width, pix.n)

    if pix.n == 4:
        img = cv2.cvtColor(img, cv2.COLOR_RGBA2BGR)
    else:
        img = cv2.cvtColor(img, cv2.COLOR_RGB2BGR)

    return img

def encode_image_to_data_url(img_bgr, quality=85):
    ok, buf = cv2.imencode(".jpg", img_bgr, [int(cv2.IMWRITE_JPEG_QUALITY), quality])
    if not ok:
        raise RuntimeError("Failed to JPEG-encode image.")
    b64 = base64.b64encode(buf.tobytes()).decode("utf-8")
    return f"data:image/jpeg;base64,{b64}"

def detect_layout_blocks(img_bgr, page_number):
    model = get_layout_model()

    det_res = model.predict(
        img_bgr,
        imgsz=YOLO_IMGSZ,
        conf=YOLO_CONF,
        device="cpu"
    )

    result = det_res[0]
    names = getattr(result, "names", {})
    boxes = result.boxes

    if boxes is None or len(boxes) == 0:
        return []

    blocks = []
    h, w = img_bgr.shape[:2]

    for i in range(len(boxes)):
        xyxy = boxes.xyxy[i].cpu().numpy().astype(int).tolist()
        cls_id = int(boxes.cls[i].item()) if boxes.cls is not None else -1
        conf = float(boxes.conf[i].item()) if boxes.conf is not None else None

        x1, y1, x2, y2 = xyxy
        x1 = max(0, min(x1, w - 1))
        y1 = max(0, min(y1, h - 1))
        x2 = max(0, min(x2, w))
        y2 = max(0, min(y2, h))

        if x2 <= x1 or y2 <= y1:
            continue

        label = names.get(cls_id, f"class_{cls_id}") if isinstance(names, dict) else str(cls_id)
        crop = img_bgr[y1:y2, x1:x2].copy()

        blocks.append({
            "page": page_number,
            "x1": x1, "y1": y1, "x2": x2, "y2": y2,
            "label": label,
            "conf": conf,
            "crop": crop
        })

    return blocks[:MAX_BLOCKS_PER_PAGE]

def page_to_markdown(page_number, pdf_path):
    doc = fitz.open(pdf_path)
    page = doc[page_number - 1]
    page_img = page_to_bgr(page, zoom=ZOOM)
    blocks = detect_layout_blocks(page_img, page_number)
    doc.close()

    if not blocks:
        return page_number, f"\n\n## Page {page_number}\n\n[No layout blocks detected]\n"

    content = []

    # Send full page first so the model can infer reading order visually
    content.append({
        "type": "input_text",
        "text": (
            "You are converting a PDF page into clean Markdown.\n\n"
            "You will receive:\n"
            "1. The full page image\n"
            "2. A set of cropped layout regions detected from that page\n\n"
            "Important:\n"
            "- Do NOT rely on the order in which the cropped regions are provided.\n"
            "- Infer the correct reading order from the full page layout and the bounding boxes.\n"
            "- Reconstruct the page in natural reading order.\n"
            "- For multi-column pages, follow the correct column-wise reading order.\n"
            "- Use the full page image as the primary source for layout understanding.\n"
            "- Use the crops for detailed text extraction.\n\n"
            "Instructions:\n"
            "1. Output clean Markdown only.\n"
            "2. Preserve headings using #/##/###.\n"
            "3. Convert tables into Markdown tables where possible.\n"
            "4. For figures/charts/images, add a detailed placeholder like [Figure: ...] including an in-depth summary without altering the original content.\n"
            "5. Ignore decorative items, page numbers, repeated headers/footers unless meaningful.\n"
            "6. Do not hallucinate missing text.\n"
            "7. If some text is unclear, keep it minimal and faithful.\n\n"
            f"Page number: {page_number}\n"
            f"Detected blocks: {len(blocks)}\n"
        )
    })

    content.append({
        "type": "input_text",
        "text": "Full page image:"
    })
    content.append({
        "type": "input_image",
        "image_url": encode_image_to_data_url(page_img)
    })

    content.append({
        "type": "input_text",
        "text": (
            "Detected blocks are provided below. Their sequence is arbitrary. "
            "Use bbox coordinates together with the full page image to determine the correct reading order."
        )
    })

    for idx, block in enumerate(blocks, start=1):
        content.append({
            "type": "input_text",
            "text": (
                f"Block {idx}\n"
                f"label={block['label']}\n"
                f"confidence={block['conf']}\n"
                f"bbox=({block['x1']},{block['y1']},{block['x2']},{block['y2']})"
            )
        })
        content.append({
            "type": "input_image",
            "image_url": encode_image_to_data_url(block["crop"])
        })

    response = client.responses.create(
        model=MODEL_NAME,
        input=[{
            "role": "user",
            "content": content
        }]
    )

    md = response.output_text.strip()
    page_md = f"\n\n## Page {page_number}\n\n{md}\n"
    return page_number, page_md

# -------------------------
# Run in parallel
# -------------------------
with fitz.open(PDF_PATH) as doc:
    total_pages = len(doc)

print(f"Total pages: {total_pages}")

results = {}
errors = []

with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
    futures = {
        executor.submit(page_to_markdown, page_num, PDF_PATH): page_num
        for page_num in range(1, total_pages + 1)
    }

    for future in as_completed(futures):
        page_num = futures[future]
        try:
            pnum, page_md = future.result()
            results[pnum] = page_md
            print(f"Done page {pnum}")
        except Exception as e:
            errors.append((page_num, str(e)))
            print(f"Failed page {page_num}: {e}")

# -------------------------
# Save combined markdown
# -------------------------
final_md_path = Path(OUTPUT_DIR) / "document.md"
with open(final_md_path, "w", encoding="utf-8") as f:
    f.write(f"# Markdown extraction for {Path(PDF_PATH).name}\n")
    for p in range(1, total_pages + 1):
        if p in results:
            f.write(results[p])
        else:
            f.write(f"\n\n## Page {p}\n\n[Extraction failed]\n")

print(f"\nSaved markdown to: {final_md_path}")

if errors:
    print("\nErrors:")
    for page_num, err in errors:
        print(f"  Page {page_num}: {err}")

C:\Users\78235\AppData\Roaming\Python\Python312\site-packages\requests\__init__.py:113: RequestsDependencyWarning: urllib3 (2.6.3) or chardet (7.1.0)/charset_normalizer (3.4.4) doesn't match a supported version!
  warnings.warn(


KeyboardInterrupt: 